# 生成式AI應用開發｜第02週
## Python 應用開發複習

> **執行環境：Google Colab** ｜ **本週不使用 LLM API**

---

Week 3 起我們會開始呼叫 OpenAI API。本週的目標**不是**從頭學 Python，而是複習在 API 開發中最常用到的語法模式。

本週每個練習都和下週的 API 開發直接相關，請確保每一格都能順利執行。

> **教師版**：本檔保留完整參考答案，可用於課堂示範、投影講解或課後核對。

## 本週學習目標

| # | 能力 | 對應 API 開發情境 |
|---|------|------------------|
| 1 | 定義與呼叫函式 | 將 API 呼叫封裝成可重複使用的函式 |
| 2 | 操作 List 與 Dict | 解析 API 回應（通常是 dict 或 list） |
| 3 | 處理 JSON | API 資料的交換格式 |
| 4 | 例外處理 | 處理 API 逾時、額度超限、格式錯誤 |
| 5 | 管理環境變數 | 在 Colab 中安全存放 API key |
| 6 | 呼叫公開 API | 為 Week 3 呼叫 OpenAI API 做準備 |

---
## 第一節：套件安裝與匯入

在 Google Colab 中，使用 `!pip install` 安裝套件。
每次重新開啟 Colab 都需要重新執行安裝。

In [ ]:
# 安裝本週需要的套件
!pip install requests --quiet
print("安裝完成")

In [ ]:
# 匯入本週使用的模組
import json      # 處理 JSON 格式
import os        # 存取環境變數
import requests  # 呼叫 HTTP API

print("模組載入完成")

---
## 第二節：函式（Function）

函式是 API 開發的基本單位。例如，下週呼叫 OpenAI API 的程式碼，
通常會封裝成函式，方便在不同地方重複呼叫，並統一管理參數。

### 2-1 基本函式定義

```python
def 函式名稱(參數1, 參數2):
    # 函式主體
    return 回傳值
```

### 2-0 f-string：字串格式化

f-string 是 Python 3.6+ 的語法，讓你在字串裡直接嵌入變數或運算式。
本教材後續大量使用，先用兩分鐘熟悉。


In [ ]:
# f-string：在字串前加 f，用大括號包住變數或運算式
name = "OpenAI"
version = 4
tokens = 1500
# token 數通常來自 API 回應；這裡先用假資料練習費用格式化。

print(f"使用 {name} 第 {version} 代模型")
print(f"共使用 {tokens} 個 token")
# :.6f 代表固定顯示 6 位小數，適合顯示很小的 API 成本。
print(f"費用估算：${tokens * 0.15 / 1_000_000:.6f} 美元")

# 舊式寫法（不推薦）供對照：
# print("使用 " + name + " 第 " + str(version) + " 代模型")


In [ ]:
# 基本函式：接收文字，回傳處理後的結果
def clean_text(text):
    """移除文字前後的空白，並轉為小寫"""
    # strip() 先移除頭尾空白；lower() 讓後續比對不受大小寫影響。
    return text.strip().lower()


# 呼叫函式
result = clean_text("  Hello World  ")
print(result)  # 輸出：hello world

# 函式可以重複呼叫
texts = ["  Python  ", " API ", "  LLM  "]
for t in texts:
    print(clean_text(t))


### 2-2 預設參數（Default Arguments）

呼叫 API 時，很多參數都有預設值（例如模型名稱、溫度值），
使用預設參數可以讓函式在大部分情況下直接呼叫，只在需要時才傳入特定值。

In [ ]:
# 預設參數：有預設值的參數放在後面
def generate_prompt(user_input, role="助理", language="繁體中文"):
    """根據使用者輸入產生 prompt 文字"""
    # role 與 language 做成參數，之後就能用同一函式產生不同 prompt。
    return f"你是一個{role}，請用{language}回答：{user_input}"


# 只傳必要參數，其他使用預設值
p1 = generate_prompt("什麼是 RAG？")
print(p1)
print()

# 覆蓋預設值
p2 = generate_prompt("What is RAG?", role="expert", language="English")
print(p2)


### 2-3 回傳多個值

Python 函式可以用 tuple 的方式回傳多個值，
在 API 開發中常用於同時回傳結果和狀態資訊。

In [ ]:
# 回傳多個值（實際上是回傳一個 tuple）
def process_response(text):
    """處理回傳文字，同時回傳結果和字數"""
    # 先清理文字，再計算長度，避免頭尾空白影響判斷。
    cleaned = text.strip()
    word_count = len(cleaned)
    is_empty = (word_count == 0)
    # Python 會把多個回傳值包成 tuple，呼叫端可以直接解包。
    return cleaned, word_count, is_empty


# 解包多個回傳值
content, length, empty = process_response("  這是 API 回傳的內容  ")
print(f"內容：{content}")
print(f"字數：{length}")
print(f"是否為空：{empty}")


### 2-4 練習：設計一個格式化訊息的函式

下週我們會傳 `messages` 給 OpenAI API，格式如下：

```python
{"role": "user", "content": "你好"}
```

請完成以下函式，讓它能快速建立這個格式：

In [ ]:
def make_message(role, content):
    """
    建立 OpenAI API 使用的訊息格式
    role: 'user' 或 'assistant' 或 'system'
    content: 訊息文字內容
    """
    # 這個 dict 結構會在下週 API messages/input 中反覆使用。
    # 說明：回傳一個包含 role 和 content 的 dict
    return {"role": role, "content": content}


# 測試
msg1 = make_message("system", "你是一個有幫助的助理")
msg2 = make_message("user", "什麼是生成式AI？")

print(msg1)
print(msg2)

# 組合成 messages 列表（下週會直接傳給 OpenAI API）
messages = [msg1, msg2]
print()
print("完整 messages 列表：")
print(messages)


---
## 第二節補充：字串常用操作

LLM 輸出的文字通常需要清理與拆解。
以下方法在 Week 3 之後幾乎每週都會用到。


In [ ]:
# LLM 輸出常需要的字串操作
text = "  生成式AI 是一種技術。\n  "

print(text.strip())                      # 移除頭尾空白與換行
print(text.strip().split("。"))          # 以句號分割成 list
print("AI" in text)                      # 包含判斷（True / False）
print(text.replace("技術", "工具"))       # 替換字串
print(text.strip().startswith("生成"))   # 開頭判斷

# 常見模式：清理 LLM 回傳文字
# 將模型輸出集中清理成函式，後續每次拿到回覆都能重複使用。
def clean_llm_output(t):
    return t.strip().replace("\n\n", "\n")

sample = "  這是模型的回答。\n\n請繼續提問。  "
print(clean_llm_output(sample))


---
## 第三節：List 與 Dict

OpenAI API 的請求和回應大量使用 List 和 Dict（以及它們的巢狀組合）。
熟悉這兩個資料結構，是解析 API 回應的基礎。

### 3-1 List 常用操作

In [ ]:
# List 基本操作
messages = []  # 空 list
# 對話紀錄通常用 list 保存，順序就是模型看到的上下文順序。

# 新增元素
messages.append({"role": "system", "content": "你是助理"})
messages.append({"role": "user", "content": "你好"})
messages.append({"role": "assistant", "content": "您好！有什麼可以幫您的？"})

print(f"訊息數量：{len(messages)}")
print(f"第一則：{messages[0]}")
print(f"最後一則：{messages[-1]}")
print()

# 切片（取出部分元素）
# 負數切片常用來保留最近幾則對話，避免 context 無限變長。
last_two = messages[-2:]
print(f"最後兩則：{last_two}")


In [ ]:
# List comprehension：用一行取出特定欄位
# 在 API 開發中常用來從回應列表中取出特定欄位

# 取出所有訊息的 role
# comprehension 適合把 API 回應中的某個欄位快速抽出來檢查。
roles = [msg["role"] for msg in messages]
print(f"所有 role：{roles}")

# 只取出 user 的訊息內容
user_contents = [msg["content"] for msg in messages if msg["role"] == "user"]
print(f"使用者訊息：{user_contents}")

# 計算每則訊息的字數
lengths = [len(msg["content"]) for msg in messages]
print(f"各訊息字數：{lengths}")


### 3-2 Dict 常用操作

API 回應通常是巢狀的 dict，例如 OpenAI 的回應格式大致如下：

```python
{
    "id": "chatcmpl-abc123",
    "model": "gpt-4o-mini",
    "choices": [
        {
            "message": {
                "role": "assistant",
                "content": "模型的回答在這裡"
            },
            "finish_reason": "stop"
        }
    ],
    "usage": {
        "prompt_tokens": 20,
        "completion_tokens": 50,
        "total_tokens": 70
    }
}
```

下面練習如何從這種巢狀結構中取出資料：

In [ ]:
# 模擬 OpenAI API 回應（下週會看到真實的版本）
mock_response = {
    "id": "chatcmpl-abc123",
    "model": "gpt-4o-mini",
    "choices": [
        {
            "message": {
                "role": "assistant",
                "content": "生成式AI是一種能夠產生文字、圖片或其他內容的人工智慧技術。"
            },
            "finish_reason": "stop"
        }
    ],
    "usage": {
        "prompt_tokens": 20,
        "completion_tokens": 35,
        "total_tokens": 55
    }
}

# 從巢狀 dict 取出回答內容
# 巢狀資料要由外往內逐層取值；[0] 代表取第一個候選回答。
content = mock_response["choices"][0]["message"]["content"]
print(f"模型回答：{content}")
print()

# 取出使用量資訊
# usage 是後續成本估算的資料來源，通常要另外取出保存。
usage = mock_response["usage"]
print(f"使用的 token 數：{usage['total_tokens']}")
print(f"  輸入：{usage['prompt_tokens']}")
print(f"  輸出：{usage['completion_tokens']}")


In [ ]:
# dict.get()：安全取值，避免 KeyError
# 當 key 不存在時，.get() 回傳 None（或指定的預設值），不會拋出錯誤

response_data = {
    "content": "這是回答",
    "model": "gpt-4o-mini"
}

# 危險的做法：key 不存在時會 KeyError
# error_field = response_data["finish_reason"]  # 這行會報錯！

# 安全的做法：使用 .get()
# .get(key, default) 能讓缺欄位時仍有可控的預設值。
content = response_data.get("content", "（無內容）")
finish_reason = response_data.get("finish_reason", "未知")  # key 不存在，回傳預設值

print(f"content：{content}")
print(f"finish_reason：{finish_reason}")

# 在 API 開發中，永遠建議用 .get() 取值
# 因為 API 回應的欄位可能因版本而異


---
## 第四節：JSON 處理

所有 LLM API 的資料都以 JSON 格式傳輸。
你需要會：
1. 將 Python dict 轉成 JSON 字串（傳送給 API）
2. 將 JSON 字串解析成 Python dict（接收 API 回應）

In [ ]:
import json

# Python dict → JSON 字串（序列化）
data = {
    "model": "gpt-4o-mini",
    "messages": [
        {"role": "user", "content": "你好"}
    ],
    "temperature": 0.7
}

# ensure_ascii=False 可保留可讀中文，不會把中文轉成 Unicode escape。
json_string = json.dumps(data, ensure_ascii=False)
print("JSON 字串：")
print(json_string)
print(f"型別：{type(json_string)}")
print()

# indent 參數讓輸出更易讀（適合除錯用）
# indent 適合除錯或教材展示；正式傳 API 時不一定需要縮排。
json_pretty = json.dumps(data, ensure_ascii=False, indent=2)
print("格式化輸出：")
print(json_pretty)


In [ ]:
# JSON 字串 → Python dict（反序列化）
# 當 API 回傳的是 JSON 字串時，用 json.loads() 解析

api_response_string = '{"id": "chatcmpl-123", "content": "這是回答", "tokens": 42}'

# loads() 只負責解析 JSON 字串；解析後才能用 dict/list 方式取值。
parsed = json.loads(api_response_string)
print(f"解析後型別：{type(parsed)}")
print(f"id：{parsed['id']}")
print(f"tokens：{parsed['tokens']}")


In [ ]:
# 實用技巧：將 API 回應印出方便除錯
def print_json(data, title=""):
    """格式化印出 dict 或 list，方便除錯"""
    if title:
        print(f"=== {title} ===")
    # default 情境下只印出格式化資料，不修改原本的 dict/list。
    print(json.dumps(data, ensure_ascii=False, indent=2))


# 測試
response = {
    "model": "gpt-4o-mini",
    "answer": "RAG 是 Retrieval-Augmented Generation 的縮寫",
    "usage": {"prompt_tokens": 15, "completion_tokens": 20}
}

print_json(response, title="API 回應")


### 4-1 從模型回應中取出結構化資料

Week 6 會學「Structured Outputs」，讓模型直接回傳 JSON 格式的資料。
以下練習解析這類回應：

In [ ]:
# 假設模型回傳了一段 JSON 格式的文字
model_output = '''
{
  "title": "2024年台灣AI發展報告",
  "summary": "本報告分析台灣在生成式AI領域的現況與挑戰",
  "keywords": ["生成式AI", "LLM", "企業應用"],
  "sentiment": "正面",
  "importance_score": 8
}
'''

# 解析模型輸出的 JSON
# 真實模型輸出不一定永遠是合法 JSON；後面會用 try/except 做安全解析。
result = json.loads(model_output)

print(f"標題：{result['title']}")
print(f"摘要：{result['summary']}")
print(f"關鍵字：{', '.join(result['keywords'])}")
print(f"情感：{result['sentiment']}")
print(f"重要度：{result['importance_score']}/10")


---
## 第五節：例外處理（Exception Handling）

呼叫外部 API 時，**永遠可能發生錯誤**：
- 網路中斷
- API key 無效
- 超過使用額度
- 回應格式非預期

例外處理讓程式在出錯時不會直接崩潰，而是能優雅地處理錯誤。

### 5-1 基本語法

In [ ]:
# 基本 try/except 結構
def safe_divide(a, b):
    try:
        # 可能失敗的程式放在 try 裡，成功時直接回傳結果。
        result = a / b
        return result
    except ZeroDivisionError:
        # 只捕捉預期錯誤，避免把其他 bug 也靜默吞掉。
        print("錯誤：除數不能為零")
        return None


print(safe_divide(10, 2))   # 正常執行
print(safe_divide(10, 0))   # 捕捉到錯誤，不會崩潰


In [ ]:
# 多種例外類型 + finally
def parse_json_safe(json_string):
    """
    安全解析 JSON 字串
    回傳 (成功與否, 結果或錯誤訊息)
    """
    try:
        # 模型若被要求輸出 JSON，第一步就是嘗試解析。
        data = json.loads(json_string)
        # 用 (success, result) 形式回傳，呼叫端就不用再寫 try/except。
        return True, data

    except json.JSONDecodeError as e:
        # JSON 格式錯誤（模型輸出格式不對時會發生）
        return False, f"JSON 解析失敗：{e}"

    except TypeError as e:
        # 傳入的不是字串時
        return False, f"型別錯誤：{e}"

    finally:
        # 不論成功或失敗都會執行（通常用來關閉連線、記錄 log）
        pass


# 測試
ok, result = parse_json_safe('{"key": "value"}')
print(f"成功：{ok}, 結果：{result}")

ok, result = parse_json_safe('這不是 JSON')
print(f"成功：{ok}, 錯誤：{result}")

ok, result = parse_json_safe(None)
print(f"成功：{ok}, 錯誤：{result}")


### 5-2 API 呼叫的錯誤處理模式

這是下週呼叫 OpenAI API 時會用到的標準模式：

In [ ]:
import time

def call_api_with_retry(prompt, max_retries=3):
    """
    呼叫 API 的標準模式，包含重試機制
    （下週換成真正的 OpenAI API 呼叫）
    """
    for attempt in range(max_retries):
        # attempt 從 0 開始，所以顯示給使用者時會加 1。
        try:
            # 模擬 API 呼叫（下週換成真正的呼叫）
            if attempt < 2:
                raise ConnectionError("模擬網路錯誤")

            # 模擬成功回應
            return {"content": f"這是對 '{prompt}' 的回答", "tokens": 42}

        except ConnectionError as e:
            print(f"第 {attempt + 1} 次嘗試失敗：{e}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # 指數退避：1秒、2秒、4秒
                # 重試等待時間逐次增加，避免網路或服務暫時不穩時立刻連續重打。
                print(f"等待 {wait_time} 秒後重試...")
                # 下一行會讓程式實際暫停（課堂上看到等待是正常現象，不是 Colab 當機）
                time.sleep(wait_time)
            else:
                print("已達最大重試次數，放棄")
                return None

        except Exception as e:
            # 捕捉未預期的錯誤
            print(f"未知錯誤：{e}")
            return None


result = call_api_with_retry("什麼是 LLM？")
if result:
    print(f"\n成功！回答：{result['content']}")


---
## 第六節：環境變數與 API Key 管理

**API key 是機密資訊，絕對不能出現在程式碼中。**

在 Google Colab 中，有兩種管理 API key 的方式：

### 方式 A：Colab Secrets（推薦）

1. 點選左側面板的 🔑 圖示（Secrets）
2. 點選「Add new secret」
3. Name 填入 `OPENAI_API_KEY`，Value 填入你的 key
4. 開啟「Notebook access」開關

In [ ]:
# 方式 A：從 Colab Secrets 讀取 API key（推薦）
# 需要先在左側 Secrets 面板設定

try:
    from google.colab import userdata
    # userdata.get() 只在 Colab 可用；本機會進入 except 提示。
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key:
        # 只顯示前幾碼確認讀取成功，不要印出完整 key。
        print(f"成功讀取 API key（前8碼）：{api_key[:8]}...")
    else:
        print("Secrets 中未找到 OPENAI_API_KEY，請先設定")
except Exception as e:
    print(f"提示：{e}")
    print("若在本機執行，請改用方式 B")


### 方式 B：將 API Key 設定為環境變數

在 Colab 上課時，建議先把 API key 放在 **Secrets**，再於程式中轉成環境變數。這樣後續程式可以統一使用 `os.getenv("OPENAI_API_KEY")` 讀取，也能避免把真正的 key 寫進 notebook。

本機開發時，也可以直接在終端機設定環境變數：

```powershell
# Windows PowerShell：只在目前視窗有效
$env:OPENAI_API_KEY="sk-..."

# Windows：寫入使用者環境變數，設定後請重新開啟終端機
setx OPENAI_API_KEY "sk-..."
```

```bash
# macOS / Linux：只在目前 shell session 有效
export OPENAI_API_KEY="sk-..."
```

設定完成後，Python 程式都用 `os.getenv("OPENAI_API_KEY")` 讀取。不要把真正的 API key commit、截圖或貼到作業中。


In [ ]:
# 方式 B：將 Colab Secrets 轉成環境變數
# 目的：讓後續程式可以統一用 os.getenv("OPENAI_API_KEY") 讀取

import os

try:
    from google.colab import userdata
    # 讀到 Secret 後轉存到環境變數，讓 OpenAI SDK 能用標準方式讀取。
    secret_key = userdata.get('OPENAI_API_KEY')

    if secret_key:
        os.environ['OPENAI_API_KEY'] = secret_key
        # 注意：這只在目前 notebook runtime 有效，不會永久寫入檔案。
        print(f"已將 Colab Secrets 轉成環境變數（前8碼）：{secret_key[:8]}...")
    else:
        print("Secrets 中未找到 OPENAI_API_KEY，請先在左側 Secrets 面板設定")

except Exception as e:
    print(f"目前可能不是在 Colab 執行：{e}")
    print("若在本機執行，請先用 PowerShell、setx 或 export 設定 OPENAI_API_KEY")

# 統一讀取方式：後續建立 OpenAI client 時也會使用這行
api_key = os.getenv('OPENAI_API_KEY')
print("環境變數已設定" if api_key else "尚未讀到 OPENAI_API_KEY")


> **補充說明：HTTP 請求方法**
>
> HTTP 有多種請求方法。本節練習使用 **GET**（取資料），
> 但 OpenAI API 底層使用 **POST**（傳送資料）。
> `openai` SDK 已封裝這一層，下週不需要手寫 `requests.post()`，
> 但閱讀官方文件時會看到 `POST /v1/chat/completions`，這是正常的。


---
## 第七節：綜合練習 — 呼叫公開 API

本節使用 [JSONPlaceholder](https://jsonplaceholder.typicode.com/)，
這是一個免費的測試用 API，**不需要 API key**，回應格式與真實 API 類似。

目標：練習發送 HTTP 請求、解析 JSON 回應、處理例外。
這些技巧下週呼叫 OpenAI API 時完全適用。

In [ ]:
import requests
import json

# 呼叫公開 API：取得一篇文章
url = "https://jsonplaceholder.typicode.com/posts/1"

# 這裡先用公開 API 練習 HTTP 流程；真實 AI API 也會有相似的請求/回應概念。
response = requests.get(url)

# 確認回應狀態碼（200 = 成功）
print(f"狀態碼：{response.status_code}")

# 解析 JSON 回應
# response.json() 會把 JSON 回應轉成 Python dict/list。
data = response.json()

print()
print("API 回應內容：")
print(json.dumps(data, ensure_ascii=False, indent=2))


In [ ]:
# 取出特定欄位
print(f"文章 ID：{data.get('id')}")
print(f"使用者 ID：{data.get('userId')}")
print(f"標題：{data.get('title')}")
print(f"內容（前50字）：{data.get('body', '')[:50]}...")


In [ ]:
# 取得多筆資料並用 list comprehension 處理
url_all = "https://jsonplaceholder.typicode.com/posts"
# 這裡未加錯誤處理是為了先看懂資料形狀；完整版本在下一個 cell。
response_all = requests.get(url_all)
all_posts = response_all.json()

print(f"共取得 {len(all_posts)} 篇文章")
print()

# 只取出前 5 篇的標題
# 先切前 5 筆再取 title，避免一次印出太多資料。
titles = [post["title"] for post in all_posts[:5]]
for i, title in enumerate(titles, 1):
    print(f"{i}. {title}")


In [ ]:
# 完整範例：包含函式封裝與例外處理的 API 呼叫

def fetch_post(post_id):
    """
    取得指定 ID 的文章
    回傳 (成功與否, 資料或錯誤訊息)
    """
    # 用 f-string 將 post_id 放進 URL，形成可重複使用的查詢函式。
    url = f"https://jsonplaceholder.typicode.com/posts/{post_id}"

    try:
        response = requests.get(url, timeout=5)  # 設定 5 秒逾時
        # timeout 可避免網路卡住時 notebook 無限等待。
        response.raise_for_status()              # 若狀態碼非 2xx，拋出例外
        # 先檢查 HTTP 狀態，再解析 JSON，錯誤會比較清楚。

        data = response.json()
        return True, data

    except requests.exceptions.Timeout:
        return False, "請求逾時，請確認網路連線"

    except requests.exceptions.HTTPError as e:
        return False, f"HTTP 錯誤：{e}"

    except requests.exceptions.ConnectionError:
        return False, "無法連線，請確認網路狀態"

    except Exception as e:
        return False, f"未知錯誤：{e}"


def summarize_post(post):
    """格式化輸出文章摘要"""
    return (
        f"ID: {post.get('id')}\n"
        f"標題: {post.get('title')}\n"
        # get() 搭配空字串預設值，避免 body 缺失時 len() 出錯。
        f"字數: {len(post.get('body', ''))} 字"
    )


# 測試正常情況
success, result = fetch_post(3)
if success:
    print(summarize_post(result))
else:
    print(f"失敗：{result}")

print()

# 測試錯誤情況（不存在的 ID）
success, result = fetch_post(9999)
if success:
    print(summarize_post(result))
else:
    print(f"失敗：{result}")


---
## 第八節：課堂練習

完成以下練習後，你就準備好進入 Week 3 的 OpenAI API 了。

In [ ]:
# ===== 練習 A =====
# 完成 build_messages() 函式
# 功能：接收 system prompt 和使用者輸入，回傳 messages 列表
# 這就是下週傳給 OpenAI API 的格式

def build_messages(system_prompt, user_input, history=None):
    """
    建立 OpenAI API 所需的 messages 列表

    參數：
        system_prompt: 系統說明，定義 AI 的角色與行為
        user_input: 使用者這次的輸入
        history: 過去的對話紀錄（list of dict），預設為 None

    回傳：
        messages: list of dict，格式為 [{"role": ..., "content": ...}, ...]
    """
    # messages 的順序很重要：system 在前，history 在中間，本次 user 在最後。
    messages = [{"role": "system", "content": system_prompt}]

    if history:
        messages.extend(history)  # 加入歷史紀錄
        # extend() 會把 history 中的多筆訊息逐筆加入，不會形成巢狀 list。

    messages.append({"role": "user", "content": user_input})

    return messages


# 測試
msgs = build_messages(
    system_prompt="你是一個 Python 教學助理，請用簡單的語言解釋技術概念。",
    user_input="什麼是 API？"
)

print(json.dumps(msgs, ensure_ascii=False, indent=2))


In [ ]:
# ===== 練習 B =====
# 解析 API 回應，計算費用
# OpenAI API 每次呼叫都會回傳 usage 資訊，可以用來估算費用

def calculate_cost(usage, model="gpt-4o-mini"):
    """
    根據 token 使用量計算費用（美元）

    gpt-4o-mini 定價（2026年）：
    - 輸入：$0.15 / 1M tokens
    - 輸出：$0.60 / 1M tokens
    """
    # 單價已換算成每 token 價格，計算時可直接乘上 token 數。
    pricing = {
        "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
        "gpt-4o":      {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
    }

    # 未知模型不硬算，避免產生看似精確但其實錯誤的成本。
    if model not in pricing:
        return None, f"不支援的模型：{model}"

    price = pricing[model]
    # prompt_tokens 是輸入成本，completion_tokens 是輸出成本，兩者單價不同。
    input_cost  = usage.get("prompt_tokens", 0)     * price["input"]
    output_cost = usage.get("completion_tokens", 0) * price["output"]
    total_cost  = input_cost + output_cost

    return total_cost, None


# 測試（模擬一次 API 呼叫的 usage）
mock_usage = {
    "prompt_tokens": 150,
    "completion_tokens": 80,
    "total_tokens": 230
}

cost, error = calculate_cost(mock_usage)
if error:
    print(f"錯誤：{error}")
else:
    print(f"本次費用：${cost:.6f} 美元")
    print(f"換算新台幣（×32）：NT${cost * 32:.4f}")


In [ ]:
# ===== 練習 C（選做・時間充裕再做）=====
# 從 JSONPlaceholder 取得多篇文章，找出字數最多的 3 篇

def get_top_posts_by_length(n=3):
    """
    取得字數最多的前 n 篇文章
    回傳文章列表（包含 id、title、字數）
    """
    try:
        # 設定 timeout，避免網路無回應時 notebook 長時間卡住。
        response = requests.get(
            "https://jsonplaceholder.typicode.com/posts",
            timeout=5
        )
        response.raise_for_status()
        # 確認 HTTP 成功後才解析 JSON，錯誤處理會更清楚。
        posts = response.json()

        # 為每篇文章加上字數欄位
        for post in posts:
            # body 可能缺失，因此用 get() 給空字串預設值。
            post["body_length"] = len(post.get("body", ""))

        # 依字數排序，取前 n 篇
        top_posts = sorted(posts, key=lambda p: p["body_length"], reverse=True)[:n]

        return True, top_posts

    except Exception as e:
        return False, str(e)


success, result = get_top_posts_by_length(3)
if success:
    print(f"字數最多的 {len(result)} 篇文章：")
    for i, post in enumerate(result, 1):
        print(f"\n{i}. ID: {post['id']}")
        print(f"   標題: {post['title']}")
        print(f"   字數: {post['body_length']}")
else:
    print(f"失敗：{result}")


---
## 本週小結

| 技能 | 本週學了什麼 | 下週用在哪裡 |
|------|------------|-------------|
| 函式 | 封裝邏輯、預設參數、多值回傳 | 封裝 API 呼叫函式 |
| List / Dict | 巢狀結構操作、`.get()`、list comprehension | 解析 API 回應 |
| JSON | `json.loads()` / `json.dumps()` | 處理 API 資料格式 |
| 例外處理 | `try/except`、多種錯誤類型、重試機制 | API 錯誤處理 |
| 環境變數 | Colab Secrets、`os.getenv()` | 安全存放 API key |
| requests | GET 請求、狀態碼、`.json()` | 與 Week 3 API 呼叫相同模式 |

---

## 下週預告：Week 3｜OpenAI API 入門

```python
# 下週第一堂課，我們會執行這段程式碼
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "你是一個有幫助的助理"},
        {"role": "user",   "content": "什麼是生成式AI？"}
    ]
)

print(response.choices[0].message.content)
```

**Week 3 前請完成：**
1. 申請 OpenAI 帳號並完成儲值（信用卡，最低 $5 美元）
2. 在 OpenAI 後台建立 API key
3. 在 Colab Secrets 設定 `OPENAI_API_KEY`

> 若無信用卡，請提前告知老師，可先使用 Gemini API 免費方案替代。